# Module 7: Model Evaluation and Comparison
**Purpose:** Review classification reports, select the two highest-F1 models, compare metrics, and display confusion matrices.

## Shared setup — load the original project datasets
This setup is included so this notebook can run independently in a fresh Google Colab session.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

base_url = 'https://raw.githubusercontent.com/Dinesh2006-Dudi/ML-Datasets/main/archive/'
matches = pd.read_csv(base_url + 'matches.csv')
deliveries = pd.read_csv(base_url + 'deliveries.csv')
batting_stats = pd.read_csv(base_url + 'batting_stats.csv')
bowling_stats = pd.read_csv(base_url + 'bowling_stats.csv')
venues = pd.read_csv(base_url + 'venues.csv')
squads = pd.read_csv(base_url + 'squads.csv')
print('Matches:', matches.shape, '| Deliveries:', deliveries.shape)

In [ ]:
# Prepare datasets using the cleaning steps from the original notebook.
matches = matches.copy()
deliveries = deliveries.copy()

for col in ['first_ings_score', 'first_ings_wkts', 'second_ings_score', 'second_ings_wkts']:
    if col in matches.columns:
        matches[col] = matches[col].fillna(matches[col].median())
for col in ['balls_left', 'wb_runs', 'wb_wickets']:
    if col in matches.columns:
        matches[col] = matches[col].fillna(0)
for col in ['player_of_the_match', 'top_scorer', 'best_bowling']:
    if col in matches.columns:
        matches[col] = matches[col].fillna('Unknown')
if 'match_winner' in matches.columns:
    matches = matches.dropna(subset=['match_winner']).copy()

for col, value in [('wicket_type', 'No Wicket'), ('player_dismissed', 'Not Out'), ('fielder', 'No Fielder')]:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna(value)
for col in ['wide', 'noballs', 'legbyes', 'byes']:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].fillna(0)

# Some later feature calculations require these numeric columns.
for col in ['runs_of_bat', 'extras']:
    if col in deliveries.columns:
        deliveries[col] = pd.to_numeric(deliveries[col], errors='coerce').fillna(0)
deliveries['total_runs'] = deliveries['runs_of_bat'] + deliveries['extras']
print('Cleaned matches:', matches.shape, '| Cleaned deliveries:', deliveries.shape)

In [ ]:
# CELL 11 — Build leakage-safe historical features

from collections import defaultdict

# Map 'balls' to 'deliveries' from the loaded dataset
balls = deliveries

# Ensure correct chronological order.
matches = matches.sort_values(["date", "match_id"]).reset_index(drop=True)

# Match-level innings summary: group first by match and team to sum up across all innings (including super overs)
innings_summary = balls.groupby(
    ["match_no", "batting_team"]
).agg(
    team_runs=("total_runs", "sum"),
    deliveries=("runs_of_bat", "count"),
    wickets=("player_dismissed", lambda x: (x != "Not Out").sum())
).reset_index()

match_innings = {}
for mid, g in innings_summary.groupby("match_no"):
    match_innings[mid] = g.set_index("batting_team").to_dict("index")

# Players who appeared for each team in each match
players_by_match_team = {}

for mid, g in balls.groupby("match_no"):
    players_by_match_team[mid] = {}

    for team_name, tg in g.groupby("batting_team"):
        batters = set(tg["striker"]) | set(tg["player_dismissed"]) # using striker/non-striker concepts
        # In your dataset, the column is 'striker' rather than 'batsman'
        players_by_match_team[mid].setdefault(team_name, {})["batters"] = batters

    for team_name, tg in g.groupby("bowling_team"):
        bowlers = set(tg["bowler"])
        players_by_match_team[mid].setdefault(team_name, {})["bowlers"] = bowlers

# Historical team statistics
team_hist = defaultdict(lambda: {
    "matches": 0,
    "wins": 0,
    "runs_for": 0,
    "runs_against": 0,
    "wickets_for": 0,
    "wickets_against": 0
})

# Historical player batting statistics
bat_hist = defaultdict(lambda: {
    "runs": 0,
    "balls": 0,
    "outs": 0,
    "matches": 0
})

# Historical player bowling statistics
bowl_hist = defaultdict(lambda: {
    "runs_conceded": 0,
    "legal_balls": 0,
    "wickets": 0,
    "matches": 0
})

def get_team_features(team_name):
    s = team_hist[team_name]
    n = s["matches"]

    return {
        "win_rate": s["wins"] / n if n else 0.5,
        "avg_runs_for": s["runs_for"] / n if n else 0,
        "avg_runs_against": s["runs_against"] / n if n else 0,
        "avg_wickets_for": s["wickets_for"] / n if n else 0,
        "avg_wickets_against": s["wickets_against"] / n if n else 0
    }

def get_player_features(team_name, match_id):
    current = players_by_match_team.get(match_id, {}).get(team_name, {})

    batters = current.get("batters", set())
    bowlers = current.get("bowlers", set())

    batting_values = []
    bowling_values = []

    for player in batters:
        s = bat_hist[player]
        if s["matches"] > 0:
            avg_runs = s["runs"] / s["matches"]
            sr = (s["runs"] / s["balls"] * 100) if s["balls"] else 0
            batting_values.append((avg_runs, sr))

    for player in bowlers:
        s = bowl_hist[player]
        if s["matches"] > 0:
            wickets_per_match = s["wickets"] / s["matches"]
            economy = (
                s["runs_conceded"] / (s["legal_balls"] / 6)
                if s["legal_balls"] else 99
            )
            bowling_values.append((wickets_per_match, economy))

    batter_avg_runs = np.mean([x[0] for x in batting_values]) if batting_values else 0
    batter_sr = np.mean([x[1] for x in batting_values]) if batting_values else 0
    bowler_avg_wickets = np.mean([x[0] for x in bowling_values]) if bowling_values else 0
    bowler_economy = np.mean([x[1] for x in bowling_values]) if bowling_values else 0

    return batter_avg_runs, batter_sr, bowler_avg_wickets, bowler_economy


feature_rows = []

for _, m in matches.iterrows():

    match_id = int(m["match_id"])
    team1 = m["team1"]
    team2 = m["team2"]

    t1 = get_team_features(team1)
    t2 = get_team_features(team2)

    p1 = get_player_features(team1, match_id)
    p2 = get_player_features(team2, match_id)

    feature_rows.append({
        "match_id": match_id,
        "stage": m["stage"],
        "venue": m["venue"],
        "team1": team1,
        "team2": team2,
        "toss_winner": m["toss_winner"],
        "toss_decision": m["toss_decision"],

        "team1_win_rate": t1["win_rate"],
        "team2_win_rate": t2["win_rate"],
        "team1_avg_runs_for": t1["avg_runs_for"],
        "team2_avg_runs_for": t2["avg_runs_for"],
        "team1_avg_runs_against": t1["avg_runs_against"],
        "team2_avg_runs_against": t2["avg_runs_against"],
        "team1_avg_wickets_for": t1["avg_wickets_for"],
        "team2_avg_wickets_for": t2["avg_wickets_for"],
        "team1_avg_wickets_against": t1["avg_wickets_against"],
        "team2_avg_wickets_against": t2["avg_wickets_against"],

        "team1_batter_avg_runs": p1[0],
        "team2_batter_avg_runs": p2[0],
        "team1_batter_sr": p1[1],
        "team2_batter_sr": p2[1],
        "team1_bowler_avg_wickets": p1[2],
        "team2_bowler_avg_wickets": p2[2],
        "team1_bowler_economy": p1[3],
        "team2_bowler_economy": p2[3],

        # Target: 1 means team1 won, 0 means team2 won
        "target": int(m["match_winner"] == team1)
    })

    # ---------------- UPDATE HISTORY AFTER FEATURE CREATION ----------------
    innings = match_innings.get(match_id, {})

    for team_name in [team1, team2]:
        opponent = team2 if team_name == team1 else team1

        own = innings.get(team_name, {})
        opp = innings.get(opponent, {})

        s = team_hist[team_name]
        s["matches"] += 1
        s["wins"] += int(m["match_winner"] == team_name)
        s["runs_for"] += own.get("team_runs", 0)
        s["runs_against"] += opp.get("team_runs", 0)
        s["wickets_for"] += opp.get("wickets", 0)
        s["wickets_against"] += own.get("wickets", 0)

    current_balls = balls[balls["match_no"] == match_id]

    for player, pg in current_balls.groupby("striker"):
        s = bat_hist[player]
        s["runs"] += pg["runs_of_bat"].sum()
        s["balls"] += (pg["wide"] == 0).sum()
        s["outs"] += (pg["player_dismissed"] == player).sum()
        s["matches"] += 1

    for player, pg in current_balls.groupby("bowler"):
        s = bowl_hist[player]

        # Byes and leg-byes are not charged to the bowler.
        runs_conceded = (
            pg["total_runs"]
            - pg["byes"]
            - pg["legbyes"]
        ).sum()

        legal_balls = (
            (pg["wide"] == 0) &
            (pg["noballs"] == 0)
        ).sum()

        wickets = pg["wicket_type"].isin(
            ["bowled", "caught", "lbw", "stumped", "caught and bowled"]
        ).sum()

        s["runs_conceded"] += runs_conceded
        s["legal_balls"] += legal_balls
        s["wickets"] += wickets
        s["matches"] += 1

model_df = pd.DataFrame(feature_rows)

print("Final modeling dataset:", model_df.shape)
display(model_df.head())

In [ ]:
# CELL 12 — Train/test split: 80% training, 20% testing

from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["target", "match_id"])
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training percentage:", len(X_train) / len(model_df) * 100)
print("Testing percentage:", len(X_test) / len(model_df) * 100)


In [ ]:
# CELL 15 — Train different ML classification models

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

# Identify column types dynamically from X_train
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Define the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=42
    ),
    "SVM": SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    random_state=42
),
}

trained_models = {}
results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by="F1-Score",
    ascending=False
).reset_index(drop=True)

display(results_df)

In [ ]:
# CELL 16 — Detailed classification report for every model

from sklearn.metrics import classification_report

for name, model in trained_models.items():
    print("\n" + "="*60)
    print(name)
    print("="*60)

    pred = model.predict(X_test)

    print(classification_report(
        y_test,
        pred,
        target_names=["Team 2 wins", "Team 1 wins"],
        zero_division=0
    ))

In [ ]:
# CELL 17 — Automatically select the two best models

top_two = results_df.head(2)

print("Two best models based on F1-score:")
display(top_two)

best_model_name = top_two.iloc[0]["Model"]
second_best_model_name = top_two.iloc[1]["Model"]

best_model = trained_models[best_model_name]
second_best_model = trained_models[second_best_model_name]

print("BEST MODEL:", best_model_name)
print("SECOND BEST MODEL:", second_best_model_name)


In [ ]:
# CELL 18 — Compare the two best models visually

comparison_long = top_two.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1-Score"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=comparison_long,
    x="Metric",
    y="Score",
    hue="Model"
)
plt.ylim(0, 1)
plt.title("Comparison of the Two Best Models")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predictions from both models
best_pred = best_model.predict(X_test)
second_pred = second_best_model.predict(X_test)

# Calculate confusion matrices
cm1 = confusion_matrix(y_test, best_pred)
cm2 = confusion_matrix(y_test, second_pred)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Best model confusion matrix
disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm1,
    display_labels=["Team 2 wins", "Team 1 wins"]
)
disp1.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Best Model: {best_model_name}")

# Second-best model confusion matrix
disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm2,
    display_labels=["Team 2 wins", "Team 1 wins"]
)
disp2.plot(ax=axes[1], cmap="Greens", colorbar=False)
axes[1].set_title(f"Second-Best Model: {second_best_model_name}")

plt.tight_layout()
plt.show()